# Day 7 — The Big Picture: AI, ML and Real-World Data
**30-Day ML & AI Mastery Plan · Shubham**

Days 1–6 gave you the tools: Python, data handling, probability, statistics, linear algebra and calculus. Today zooms out to answer the questions that decide whether an ML project succeeds **before any model is trained**:
- What kind of problem is this, and is ML even the right tool?
- What exactly are we predicting, from what, and how will we know it works?
- How do text, images and sound become numbers a model can use?
- Where does the data come from, and how do we clean it?

The live demo collects real data from the web, cleans it and visualises it.

| Part | What you'll learn | Time |
|---|---|---|
| 1 | AI vs ML vs deep learning vs generative AI | 15 min |
| 2 | Supervised, unsupervised and reinforcement learning (with mini demos) | 30 min |
| 3 | Framing real business problems as ML tasks | 30 min |
| 4 | Representing real-world data: tables, text, images, audio | 35 min |
| 5 | Processing data properly: pipelines | 20 min |
| **6** | **Live demo: scrape a website → clean → visualise** | 45 min |
| 7 | Exercises + Week 1 checkpoint | 30–60 min |

In [ ]:
import time, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
rng = np.random.default_rng(seed=7)

---
# Part 1 — AI vs ML vs deep learning vs generative AI

They're nested circles, each one inside the last:

| Term | Meaning | Example |
|---|---|---|
| **Artificial Intelligence** | any technique that makes computers act intelligently, including hand-written rules | a chess engine's search, an `if "free money" in email` filter |
| **Machine Learning** | systems that **learn the rules from data** instead of being told them | TextPulse's spam model, learned from 35,000 emails |
| **Deep Learning** | ML with multi-layer neural networks that also learn their own features | image recognition, speech-to-text |
| **Generative AI** | deep models that **create** new text, images, audio or code | ChatGPT, Claude, image generators |

**Traditional programming:** rules + data → answers.
**Machine learning:** data + answers → **rules** (the trained model), which then produce answers for new data.

See the difference on the same task:

In [ ]:
emails = ["WIN a FREE prize now!!!", "Team meeting moved to Friday", "Free lunch for the team on Friday",
          "Claim your cash reward today", "Your invoice for October is attached", "Exclusive offer: act now"]
labels = ["spam", "ham", "ham", "spam", "ham", "spam"]

# Traditional programming: a human writes the rule
rule_based = ["spam" if ("free" in e.lower() or "win" in e.lower()) else "ham" for e in emails]

# Machine learning: the rule is learned from labelled examples
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
learned = make_pipeline(TfidfVectorizer(), LogisticRegression()).fit(emails, labels)

for e, truth, r in zip(emails, labels, rule_based):
    print(f"{e:<40} truth={truth:<4}  hand rule={r:<4}  {'✓' if r == truth else '✗'}")
print("\nThe hand rule wrongly flags 'Free lunch for the team' and misses two real spams that don't say 'free' or 'win'.")
print("Learned model on a NEW email:", learned.predict(["Act now to claim your exclusive reward"])[0])

With 6 emails the learned model is a toy, but the principle scales: nobody could hand-write rules for 1.6 million tweets, but a model learned them in a minute (TextPulse). **Use ML when the rules are too many, too fuzzy or too fast-changing to write by hand, and you have data with examples of the answer.**

---
# Part 2 — The three main ways machines learn

| Type | Data | Goal | Examples | Algorithms (week) |
|---|---|---|---|---|
| **Supervised** | inputs **with** correct answers (labels) | predict the answer for new inputs | spam, churn, house price, diagnosis | regression, trees, SVM, neural nets (Weeks 2–3) |
| ↳ classification | label is a category | which class? | spam / ham, fraud / genuine | |
| ↳ regression | label is a number | how much? | price, delivery time | |
| **Unsupervised** | inputs **only** | find structure | customer segments, anomalies, compression | k-means, PCA (Day 12–13) |
| **Reinforcement** | an environment that gives **rewards** | learn actions that maximise reward | games, robotics, ad bidding, recommendations | Q-learning, policy gradients |

Two modern hybrids: **semi-supervised** (a few labels, lots of unlabelled data) and **self-supervised** (the data labels itself, e.g. "predict the next word": that's how LLMs are pre-trained).

Three tiny demos make the difference concrete.

In [ ]:
# SUPERVISED: learn from labelled tumours, then predict unseen ones
from sklearn.datasets import load_breast_cancer, load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
X, y = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_tr, y_tr)
print(f"Supervised: accuracy on unseen tumours = {clf.score(X_te, y_te):.3f}")

In [ ]:
# UNSUPERVISED: k-means is never shown the species. Does it rediscover them?
from sklearn.cluster import KMeans
iris = load_iris()
clusters = KMeans(n_clusters=3, n_init=10, random_state=7).fit_predict(StandardScaler().fit_transform(iris.data))
print("Unsupervised: clusters found vs real species (never shown to the algorithm)")
print(pd.crosstab(pd.Series(iris.target_names[iris.target], name="species"), pd.Series(clusters, name="cluster")))

Without ever seeing a label, k-means isolates **setosa perfectly** (all 50 in one cluster). Versicolor and virginica are only partly separated, because those two species genuinely overlap in their measurements. That's typical: unsupervised learning finds the structure that's really in the data, which isn't always the structure you hoped for.

### Reinforcement learning in 15 lines: the multi-armed bandit
An app has 3 banner designs with unknown click rates. The agent shows one per visitor, sees whether it was clicked (the **reward**), and must balance **exploring** (trying designs to learn) with **exploiting** (showing the best one so far). The ε-greedy strategy explores 10% of the time.

In [ ]:
true_click_rates = [0.03, 0.05, 0.08]          # unknown to the agent
estimates, counts = np.zeros(3), np.zeros(3)
total_clicks, epsilon = 0, 0.1
for visitor in range(10_000):
    arm = rng.integers(3) if rng.random() < epsilon else int(np.argmax(estimates))   # explore or exploit
    reward = rng.random() < true_click_rates[arm]                                     # did they click?
    counts[arm] += 1
    estimates[arm] += (reward - estimates[arm]) / counts[arm]                          # running average
    total_clicks += reward
print("times each banner was shown:", counts.astype(int))
print("estimated click rates      :", estimates.round(3), " true:", true_click_rates)
print(f"clicks: {total_clicks} vs about {int(10_000 * np.mean(true_click_rates))} if we chose randomly")

---
# Part 3 — Framing a business problem as an ML task

Most failed ML projects don't fail because of the algorithm. They fail because the problem was framed badly: the wrong target, inputs that won't exist at prediction time, a metric nobody cares about, or no plan for what happens with the prediction.

### The framing checklist
1. **Business goal**: what decision will change? (not "use AI")
2. **Prediction target**: exactly what, for whom, and **when** is it predicted?
3. **ML task type**: classification, regression, ranking, clustering, forecasting…
4. **Inputs**: only data **available at prediction time** (no leakage, Day 2).
5. **Labels**: where do historical answers come from? How many? How noisy?
6. **Metrics**: an ML metric *and* a business metric. Which error is more costly?
7. **Baseline**: what happens today, or the simplest rule? The model must beat it.
8. **Action and deployment**: who uses the prediction, how often, and how fast?

**When NOT to use ML:** the rules are simple and stable; there's no data or no labels; mistakes are unacceptable and can't be explained; or a simple report answers the question.

In [ ]:
from dataclasses import dataclass, fields

@dataclass
class MLFraming:
    problem: str
    business_goal: str
    target: str
    task: str
    inputs: str
    labels: str
    ml_metric: str
    business_metric: str
    baseline: str
    action: str
    watch_out: str

    def show(self):
        print("=" * 90); print(self.problem.upper()); print("=" * 90)
        for f in fields(self)[1:]:
            print(f"{f.name.replace('_', ' '):<16}: {getattr(self, f.name)}")
        print()

problems = [
    MLFraming(
        problem="1. Telecom customer churn",
        business_goal="Keep customers by offering retention deals to the right people before they leave",
        target="Will this customer cancel within the next 30 days? (predicted on the 1st of each month)",
        task="Binary classification (probability of churn)",
        inputs="Tenure, plan, monthly bill, data usage trend, complaints in last 90 days, payment delays",
        labels="Historical cancellations: customers active on day X who left by day X+30",
        ml_metric="Recall and precision at the top 10% of scores; PR-AUC (churn is only ~5% of customers)",
        business_metric="Revenue saved minus cost of retention offers",
        baseline="Target everyone whose contract ends this month",
        action="Monthly list of the highest-risk customers sent to the retention team",
        watch_out="'Cancellation request date' leaks the answer; offers change future behaviour (feedback loop)"),
    MLFraming(
        problem="2. Supplier delivery delay (supply chain)",
        business_goal="Flag purchase orders likely to arrive late so planners can expedite or re-plan early",
        target="Will this purchase order arrive more than 3 days after its promised date? (scored when placed)",
        task="Binary classification; could also be regression on days late",
        inputs="Supplier's past on-time %, lead time, order size, part category, route, season, port congestion",
        labels="Actual vs promised receipt dates from past purchase orders in the ERP",
        ml_metric="Recall on late orders (missing a delay is costly) at an acceptable precision",
        business_metric="Fewer line stoppages and expediting costs; planner hours saved",
        baseline="Flag every order from suppliers with on-time rate below 80%",
        action="Daily risk score in the planning dashboard; high-risk orders get a follow-up",
        watch_out="Use only information known when the order is placed; supplier mix changes over time (drift)"),
    MLFraming(
        problem="3. Food-delivery time estimate",
        business_goal="Show customers an honest delivery time to reduce cancellations and complaints",
        target="Minutes from order placement to delivery, predicted at checkout",
        task="Regression (a number); ideally with a range (prediction interval)",
        inputs="Distance, restaurant's recent prep times, time of day, weather, riders available nearby",
        labels="Actual delivery durations of past orders",
        ml_metric="Mean absolute error in minutes; % of deliveries within the promised window",
        business_metric="Cancellation rate, late-delivery complaints, repeat orders",
        baseline="Restaurant's average delivery time",
        action="Real-time estimate shown in the app (needs a fast, always-on API)",
        watch_out="Under-promising is cheaper than over-promising: optimise an asymmetric loss"),
]
for p in problems:
    p.show()

Notice how the same checklist turns three vague wishes into specific, testable tasks, and how each one's "watch out" line is a lesson from Days 2–4 (leakage, drift, the cost of different errors, imbalanced classes).

---
# Part 4 — Representing real-world data as numbers

Every model takes numbers in. The job is to turn each kind of data into numbers **without losing the information that matters**.

## 4a. Tabular data
| Column type | Example | Becomes | Watch out |
|---|---|---|---|
| numeric | age, fare | the number (scaled for many models) | outliers, skew |
| categorical, no order | port, city | **one-hot** columns | many categories → many columns |
| ordinal | small < medium < large | ordered integers 0, 1, 2 | don't one-hot away the order |
| date / time | order date | day of week, month, hour, "days since" | never feed raw timestamps |
| ID | customer_id | **drop it** | it memorises, doesn't generalise |

In [ ]:
orders = pd.DataFrame({
    "order_id": [101, 102, 103, 104],
    "city": ["Chennai", "Hyderabad", "Chennai", "Pune"],
    "size": ["small", "large", "medium", "small"],
    "amount": [250.0, 1200.0, 640.0, 180.0],
    "ordered_at": pd.to_datetime(["2026-10-30 09:15", "2026-10-31 20:40", "2026-11-01 13:05", "2026-11-01 22:30"]),
})
features = pd.get_dummies(orders[["city"]], prefix="city", dtype=int)                    # one-hot
features["size"] = orders["size"].map({"small": 0, "medium": 1, "large": 2})               # ordinal
features["log_amount"] = np.log1p(orders["amount"]).round(3)                               # skewed money
features["hour"] = orders["ordered_at"].dt.hour                                            # date parts
features["is_weekend"] = (orders["ordered_at"].dt.dayofweek >= 5).astype(int)
features                                                                                   # order_id dropped

## 4b. Text
Text becomes a vector. Three levels, in order of power:
1. **Bag of words / counts**: one column per word, the count in each document.
2. **TF-IDF**: counts re-weighted so rare, distinctive words matter more (TextPulse, Day 5).
3. **Embeddings**: dense vectors from a neural network where similar meanings are close together (Days 22–28).

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
docs = ["the delivery was late", "late delivery again", "great food and fast delivery"]
cv = CountVectorizer()
counts = cv.fit_transform(docs)
pd.DataFrame(counts.toarray(), columns=cv.get_feature_names_out(), index=docs)

Each document is now a row of numbers: a vector in "word space". Most entries are 0, which is why text matrices are stored **sparse** (only the non-zeros), as TextPulse does with its 200,000 columns.

## 4c. Images
An image is a grid of pixel intensities: **(height, width)** for greyscale, **(height, width, 3)** for colour (red, green, blue, each 0–255).

In [ ]:
from sklearn.datasets import load_digits, load_sample_image
digits = load_digits()
photo = load_sample_image("china.jpg")
print("one handwritten digit:", digits.images[0].shape, "| a colour photo:", photo.shape, photo.dtype)
print("the digit's 8x8 pixel values (0 = white, 16 = black):\n", digits.images[0].astype(int))

fig, axes = plt.subplots(1, 5, figsize=(16, 3))
axes[0].imshow(digits.images[0], cmap="gray_r"); axes[0].set_title(f"digit '{digits.target[0]}' (8x8)")
axes[1].imshow(photo); axes[1].set_title("photo (427x640x3)")
for i, (ch, cmap) in enumerate([("red", "Reds"), ("green", "Greens"), ("blue", "Blues")]):
    axes[i + 2].imshow(photo[..., i], cmap=cmap); axes[i + 2].set_title(f"{ch} channel")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()
print("flattened for a classic model: the digit becomes a vector of", digits.images[0].size, "numbers")

Flattening throws away which pixels are neighbours. **Convolutional neural networks** (Day 18) keep the 2-D layout and learn edges, textures and shapes, which is why they dominate vision.

## 4d. Audio
Sound is air pressure measured thousands of times per second: a **1-D array** (a waveform). 16,000 samples per second is common for speech. Models usually work on a **spectrogram**: how much of each frequency is present over time, which turns sound into an image.

In [ ]:
from scipy import signal
sr = 16_000                                         # samples per second
t = np.arange(0, 1.0, 1 / sr)
tone = np.where(t < 0.5, np.sin(2 * np.pi * 440 * t), np.sin(2 * np.pi * 880 * t))   # A4, then A5
audio = tone + 0.3 * rng.normal(size=t.size)        # add background noise
print("1 second of audio =", audio.shape[0], "numbers")

freqs, times, spec = signal.spectrogram(audio, fs=sr, nperseg=512)
fig, axes = plt.subplots(1, 2, figsize=(15, 3.5))
axes[0].plot(t[:800], audio[:800]); axes[0].set(title="Waveform (first 0.05 s)", xlabel="seconds", ylabel="pressure")
axes[1].pcolormesh(times, freqs, 10 * np.log10(spec + 1e-12), shading="auto", cmap="magma")
axes[1].set(ylim=(0, 2000), title="Spectrogram: 440 Hz, then 880 Hz, plus noise", xlabel="seconds", ylabel="frequency (Hz)")
plt.tight_layout(); plt.show()

| Data | Raw form | Classic ML input | Deep learning input |
|---|---|---|---|
| Tabular | rows and columns | scaled numbers + one-hot | the same (trees usually win on tables) |
| Text | strings | counts / TF-IDF | token IDs → embeddings (transformers) |
| Image | H × W × 3 pixels | flattened pixels or hand-made features | the pixel grid (CNNs, vision transformers) |
| Audio | 1-D waveform | spectrogram statistics (MFCCs) | spectrogram or raw waveform |
| Time series | values over time | lags, rolling averages, date parts | the sequence (RNNs, transformers) |

---
# Part 5 — Processing data properly: one pipeline for every column

Real tables mix column types. scikit-learn's **ColumnTransformer** applies the right steps to each group, and a **Pipeline** chains it with the model, so every step is **fitted on training data only** (Day 4) and applied identically to new data in production.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

titanic = sns.load_dataset("titanic")
numeric = ["age", "fare", "sibsp", "parch"]
categorical = ["sex", "embarked", "pclass"]
X = titanic[numeric + categorical]
y = titanic["survived"]

preprocess = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numeric),
    ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")), categorical),
])
model = make_pipeline(preprocess, LogisticRegression(max_iter=1000))

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model.fit(X_tr, y_tr)
print(f"test accuracy: {model.score(X_te, y_te):.3f}   (baseline 'everyone died': {1 - y_te.mean():.3f})")
print("columns after preprocessing:", list(model[0].get_feature_names_out()))

Seven raw columns with missing values became 12 clean numeric features, and the whole thing is **one object**: save it with joblib, load it in an API, and call `predict` on raw rows. That's exactly how TextPulse's TF-IDF + model pipeline is deployed.

---
# Part 6 — LIVE DEMO: scrape a website → clean → visualise

Not every dataset comes as a CSV. **Web scraping** means downloading web pages and extracting data from their HTML.

### Rules first (scraping responsibly)
- Prefer an **official API or download** if one exists.
- Read the site's **Terms of Service** and **robots.txt** (`site.com/robots.txt` lists what bots may visit).
- Be **polite**: identify yourself, pause between requests, and don't hammer a server.
- Don't collect **personal data** without a lawful basis (India's DPDP Act, Europe's GDPR).

We'll use **books.toscrape.com**, a sandbox built specifically for scraping practice: 1,000 fictional books in 50 categories.

### HTML in 30 seconds
A page is nested **tags**, and each tag can have attributes such as `class`. Right-click → **Inspect** in your browser to see them. One book on this site looks like this (simplified):
```html
<article class="product_pod">
  <p class="star-rating Three"></p>
  <h3><a href="catalogue/a-light-in-the-attic_1000/index.html" title="A Light in the Attic">A Light in the ...</a></h3>
  <p class="price_color">£51.77</p>
  <p class="instock availability">In stock</p>
</article>
```
Note the details: the full title is in the `title` attribute (the visible text is cut short), and the rating is hidden in a **class name**, not in the text.

In [ ]:
from bs4 import BeautifulSoup
import requests
from urllib.parse import urljoin

card_html = '''<article class="product_pod">
  <p class="star-rating Three"><i class="icon-star"></i></p>
  <h3><a href="catalogue/a-light-in-the-attic_1000/index.html" title="A Light in the Attic">A Light in the ...</a></h3>
  <div class="product_price"><p class="price_color">£51.77</p>
  <p class="instock availability"><i class="icon-ok"></i> In stock </p></div>
</article>'''

def parse_card(card, page_url, category):
    """Pull the fields we want out of one <article class="product_pod">."""
    link = card.select_one("h3 a")
    return {
        "title": link["title"],                                                # full title is in the attribute
        "price_raw": card.select_one("p.price_color").get_text(strip=True),
        "rating_raw": card.select_one("p.star-rating")["class"][1],           # ['star-rating', 'Three'] -> 'Three'
        "availability": card.select_one("p.availability").get_text(strip=True),
        "category": category,
        "url": urljoin(page_url, link["href"]),                               # relative link -> full URL
    }

card = BeautifulSoup(card_html, "html.parser").select_one("article.product_pod")
parse_card(card, "https://books.toscrape.com/", "Poetry")

### A classic bug: the "Â£" problem
This site's server doesn't say which text encoding it uses, so `requests` guesses ISO-8859-1 and the pound sign arrives as **Â£**. Here's exactly what goes wrong, and the one-line fix:

In [ ]:
price_bytes = "£51.77".encode("utf-8")            # what the server actually sends
print("decoded with the wrong guess:", price_bytes.decode("latin-1"))
print("decoded correctly           :", price_bytes.decode("utf-8"))
print("fix in requests             : response.encoding = 'utf-8'  (before reading response.text)")

### The crawler
Plan: read the home page → collect the 50 category links → visit each category, following the **"next"** link until there isn't one → parse every book card. A short pause between requests keeps us polite. In Colab this takes about a minute.

If the site can't be reached (no internet, or a blocked network), the notebook falls back to a built-in sample of 20 real books from the site so the rest still runs; the findings below are from the full 1,000-book scrape.

In [ ]:
BASE = "https://books.toscrape.com/"
HEADERS = {"User-Agent": "ml-30-days-learning-notebook (educational)"}

def get_soup(session, url):
    response = session.get(url, headers=HEADERS, timeout=15)
    response.raise_for_status()                    # stop on 404 / 500 errors
    response.encoding = "utf-8"                    # the Â£ fix
    return BeautifulSoup(response.text, "html.parser")

def scrape_all_books(pause=0.2):
    rows = []
    with requests.Session() as session:            # reuses one connection: faster and kinder to the server
        home = get_soup(session, BASE)
        categories = [(urljoin(BASE, a["href"]), a.get_text(strip=True))
                      for a in home.select(".side_categories ul li ul li a")]
        for cat_url, cat_name in categories:
            page_url = cat_url
            while page_url:                        # follow "next" links through the pages
                soup = get_soup(session, page_url)
                rows += [parse_card(c, page_url, cat_name) for c in soup.select("article.product_pod")]
                nxt = soup.select_one("ul.pager li.next a")
                page_url = urljoin(page_url, nxt["href"]) if nxt else None
                time.sleep(pause)                  # be polite
    return pd.DataFrame(rows)

SAMPLE = [  # 20 real books from the site, used only if it can't be reached (titles shortened)
    ("It's Only the Himalayas", "£45.17", "Two", "Travel"), ("Sharp Objects", "£47.82", "Four", "Mystery"),
    ("Tipping the Velvet", "£53.74", "One", "Historical Fiction"), ("The Last Painting of Sara de Vos", "£55.55", "Two", "Historical Fiction"),
    ("Lumberjanes, Vol. 2: Friendship to the Max", "£46.91", "Two", "Sequential Art"), ("Persepolis: The Story of a Childhood", "£39.13", "One", "Sequential Art"),
    ("Run, Spot, Run: The Ethics of Keeping Pets", "£20.02", "One", "Philosophy"), ("The Devil Wears Prada", "£44.29", "One", "Womens Fiction"),
    ("Miller's Valley", "£58.54", "Two", "Fiction"), ("#HigherSelfie: Wake Up Your Life", "£23.11", "Five", "Nonfiction"),
    ("Smarter Faster Better", "£38.89", "Five", "Nonfiction"), ("The Coming Woman: A Novel", "£17.93", "Three", "Default"),
    ("The Martian (The Martian #1)", "£41.39", "Two", "Default"), ("Jane Eyre", "£38.43", "Five", "Default"),
    ("A Brush of Wings (Angels Walking #3)", "£55.51", "One", "Add a comment"), ("The Star-Touched Queen", "£46.02", "Five", "Fantasy"),
    ("Catching Jordan (Hundred Oaks)", "£50.83", "Three", "Young Adult"), ("Night Sky with Exit Wounds", "£41.05", "One", "Poetry"),
    ("America's War for the Greater Middle East", "£51.22", "Two", "History"), ("You Are a Badass", "£12.08", "Three", "Self Help"),
]

started = time.perf_counter()
try:
    raw = scrape_all_books()
    source = "live scrape"
except requests.RequestException as error:
    print("Couldn't reach the site:", type(error).__name__, "-> using the built-in 20-book sample")
    raw = pd.DataFrame(SAMPLE, columns=["title", "price_raw", "rating_raw", "category"])
    raw["availability"], raw["url"] = "In stock", None
    source = "built-in sample"
print(f"{source}: {len(raw)} books in {time.perf_counter() - started:.1f}s")
raw.head()

### Clean
Everything scraped is **text**. Cleaning turns it into analysable columns, and checks quality:

In [ ]:
books = raw.copy()
books["price"] = books["price_raw"].str.replace(r"[^0-9.]", "", regex=True).astype(float)   # '£51.77' -> 51.77
books["rating"] = books["rating_raw"].map({"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5})

print("duplicate URLs        :", books["url"].duplicated().sum() if books["url"].notna().any() else "n/a")
print("missing values        :", int(books[["title", "price", "rating"]].isna().sum().sum()))
print("availability values   :", books["availability"].value_counts().to_dict())
print("categories            :", books["category"].nunique())

Two quality findings, both typical of real scraped data:
1. **Every book says "In stock"**: a column with one value carries zero information (Day 3: no dependence on anything). Drop it. (Stock *counts* exist only on each book's own page, which would take 1,000 more requests.)
2. **Junk categories**: "Default" (152 books) and "Add a comment" (67 books) aren't genres; they're placeholder values left in the source data. Flag them rather than treat them as real categories.

In [ ]:
books = books.drop(columns=["availability", "price_raw", "rating_raw"])
books["category_is_junk"] = books["category"].isin(["Default", "Add a comment"])
print(f"junk-category books: {books['category_is_junk'].sum()} of {len(books)}")
books.to_csv("books_clean.csv", index=False)            # save the clean dataset
books.describe().round(2)

### Visualise and conclude

In [ ]:
real = books[~books["category_is_junk"]]
big_cats = real["category"].value_counts()
big_cats = big_cats[big_cats >= 20].index if len(books) > 100 else big_cats.index    # categories with enough books

fig, axes = plt.subplots(2, 2, figsize=(15, 9))
sns.histplot(books["price"], bins=25, ax=axes[0, 0]); axes[0, 0].set_title("Price distribution (£)")
sns.countplot(x=books["rating"], ax=axes[0, 1]); axes[0, 1].set_title("How many books per star rating")
order = real[real["category"].isin(big_cats)].groupby("category")["price"].median().sort_values().index
sns.boxplot(data=real[real["category"].isin(big_cats)], y="category", x="price", order=order, ax=axes[1, 0])
axes[1, 0].set_title("Price by category (categories with 20+ books)")
sns.stripplot(data=books, x="rating", y="price", alpha=0.35, jitter=0.25, ax=axes[1, 1])
r = books["price"].corr(books["rating"])
axes[1, 1].set_title(f"Price vs rating: correlation r = {r:.3f}")
plt.tight_layout(); plt.show()

print(f"price: mean £{books['price'].mean():.2f}, median £{books['price'].median():.2f}, range £{books['price'].min():.2f}–£{books['price'].max():.2f}")
print("books per rating:", books["rating"].value_counts().sort_index().to_dict())

### Findings (from the full 1,000-book scrape)
1. **Prices are spread evenly** between £10 and £60 (mean £35.07, median £35.98), with no skew: unusual for real prices, a hint that this is generated data.
2. **Ratings are balanced** across 1–5 stars (179 to 226 books each), unlike real stores, where most ratings are 4–5 stars.
3. **Rating doesn't predict price**: correlation r = 0.03. A model trying to predict price from rating would learn nothing.
4. **Category matters a little**: among categories with 20+ books, Fantasy has the highest average price (£39.59) and Food and Drink the lowest (£31.41).
5. **Data quality**: 219 books (22%) sit in placeholder categories, and availability carries no information.

That's the real value of EDA on scraped data: before any model, you've learned which fields are usable, which are junk, and that one obvious hypothesis (better-rated books cost more) is false here.

### Bonus: tables are the easiest thing to scrape
If the data is already an HTML `<table>`, pandas reads it in one line (needs `lxml`, pre-installed in Colab):
```python
tables = pd.read_html("https://en.wikipedia.org/wiki/List_of_states_and_union_territories_of_India_by_population")
tables[0].head()
```
For pages built by JavaScript (where the HTML arrives empty and fills in later), use a browser-automation tool such as **Playwright** or **Selenium**, or look in the browser's Network tab for the JSON the page loads, which is often an easier source than the HTML.

---
# Part 7 — Exercises
Paste your code in chat for review.

**Ex 1 — Classify the problem.** For each, name the learning type and task (classification / regression / clustering / RL / not ML) and one metric: (a) group 50,000 shoppers into segments for marketing; (b) predict tomorrow's electricity demand per hour; (c) decide which of 4 discount offers to show each visitor and improve from responses; (d) flag invoices whose amount doesn't match the purchase order; (e) estimate the age of a person from a photo.

**Ex 2 — Frame your own.** Pick one problem from your work in supply chain or software delivery. Fill in an `MLFraming` for it, including a baseline and one leakage risk.

**Ex 3 — Scrape more.** Extend `parse_card` to also scrape the **star rating and price from each book's own page** for one category, plus the **stock count** (e.g. "In stock (22 available)") using a regular expression. Is stock count related to price?

**Ex 4 — Quotes.** Scrape all pages of **quotes.toscrape.com** (quote text, author, tags; follow the "Next" button). Which 10 tags are most common, and which author has the most quotes? Plot both.

**Week 1 checkpoint.** If you haven't yet: one notebook that loads a raw CSV, cleans it, standardises a feature and fits a line with your own gradient descent (Day 6, Week 1 checkpoint). This closes Week 1.

In [ ]:
# Ex 1 (answers as comments or a small DataFrame)

In [ ]:
# Ex 2

In [ ]:
# Ex 3

In [ ]:
# Ex 4

---
### Day 7 recap
- **AI ⊃ ML ⊃ deep learning ⊃ generative AI.** ML learns rules from data plus answers; use it when the rules are too many or fuzzy to write.
- **Supervised** (labels: classification or regression), **unsupervised** (structure: clusters, PCA), **reinforcement** (rewards: explore vs exploit). LLMs are pre-trained **self-supervised**.
- **Framing checklist:** goal → target (and when) → task → inputs available at prediction time → labels → ML and business metrics → baseline → action. Most project failures are framing failures.
- **Representation:** tables (scale, one-hot, ordinal, date parts, drop IDs), text (counts → TF-IDF → embeddings), images (H×W×3 arrays), audio (waveform → spectrogram).
- **ColumnTransformer + Pipeline:** the right preprocessing per column, fitted on train only, deployable as one object.
- **Scraping:** check the rules and robots.txt, be polite, parse with CSS selectors, watch encodings (Â£), follow pagination, then clean: constant columns and placeholder categories are common.
- **Week 1 complete.** Next week: classical ML, starting with **Day 8: linear and logistic regression and k-NN**.